# Task 14 - Product Text NLP Analysis

**Goal:** analyze catalog title/tag language and its association with product ratings.

**Input:** `data/processed/womens_clothing_final.csv`
**Outputs:** product text features and reports in `data/processed/` and `reports/`.

> No review-text column exists in the available dataset. This analysis uses product titles and tags, and does not treat them as customer reviews.

In [1]:
from pathlib import Path
from datetime import datetime
import re
import numpy as np
import pandas as pd

def find_project_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "processed").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate a project folder containing data/processed.")

ROOT = find_project_root()
PROC = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)
DATA_PATH = PROC / "womens_clothing_final.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Expected product catalog CSV at {DATA_PATH}")
df = pd.read_csv(DATA_PATH)
df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")].copy()
print(f"Project: {ROOT}\nLoaded: {DATA_PATH.name} | rows={len(df):,} | columns={len(df.columns)}")

def require_columns(frame, columns):
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise KeyError(f"{DATA_PATH.name} is not the expected product-catalog dataset. "
                       f"Missing columns: {missing}. Available columns: {list(frame.columns)}")

from scipy import stats
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

require_columns(df, ["title", "tags", "rating"])
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df = df.dropna(subset=["rating"]).copy()
df["product_text"] = (df["title"].fillna("").astype(str) + " " +
                      df["tags"].fillna("").astype(str)).str.strip()
print(f"Products with title/tag text: {(df['product_text'].str.len() > 0).sum():,} of {len(df):,}")

Project: D:\OneDrive\Desktop\Brainybeams\Womens_Clothing_ML_Project
Loaded: womens_clothing_final.csv | rows=1,539 | columns=40
Products with title/tag text: 1,539 of 1,539


In [2]:
analyzer = SentimentIntensityAnalyzer()
word_pattern = re.compile(r"[A-Za-z']+")
text = df["product_text"].astype(str)
words = text.map(word_pattern.findall)
sentiment = text.map(lambda value: analyzer.polarity_scores(value)["compound"])
features = pd.DataFrame(index=df.index)
features["txt_char_count"] = text.str.len()
features["txt_word_count"] = words.map(len)
features["txt_avg_word_length"] = words.map(
    lambda tokens: float(np.mean([len(token) for token in tokens])) if tokens else 0.0
)
features["txt_lexical_diversity"] = words.map(
    lambda tokens: len({token.casefold() for token in tokens}) / max(len(tokens), 1)
)
features["txt_exclamation_count"] = text.str.count("!")
features["txt_question_count"] = text.str.count(r"\?")
features["txt_sentiment_compound"] = sentiment
features["txt_title_word_count"] = df["title"].fillna("").astype(str).str.split().str.len()
features["high_rating"] = (df["rating"] >= 4).astype(int)

correlations = []
for column in features.columns.drop("high_rating"):
    if features[column].nunique(dropna=True) < 2:
        continue
    value, p_value = stats.spearmanr(features[column], df["rating"])
    correlations.append({"feature": column, "spearman_rho": value, "spearman_p": p_value})
correlations = pd.DataFrame(correlations)
display(correlations.sort_values("spearman_rho", key=lambda values: values.abs(),
                                 ascending=False).round(4))

,feature,spearman_rho,spearman_p
2,txt_avg_word_length,0.1416,0.0000
5,txt_sentiment_compound,0.1017,0.0001
3,txt_lexical_diversity,-0.0275,0.2811
0,txt_char_count,0.0254,0.3184
1,txt_word_count,-0.0241,0.3449
4,txt_exclamation_count,-0.0145,0.5707
6,txt_title_word_count,-0.0018,0.9437


In [3]:
tfidf = TfidfVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2), max_features=5000)
text_matrix = tfidf.fit_transform(text)
model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
labels = features["high_rating"]
if labels.value_counts().min() < 5:
    raise ValueError("Need at least five examples in each rating class for 5-fold evaluation.")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_scores = cross_val_score(model, text_matrix, labels, cv=cv, scoring="roc_auc")
model.fit(text_matrix, labels)
terms = np.asarray(tfidf.get_feature_names_out())
coefficients = model.coef_[0]
term_table = pd.DataFrame({"term": terms, "rating_association": coefficients})
term_table = term_table.sort_values("rating_association", ascending=False)
print(f"Text-only 5-fold ROC-AUC: {auc_scores.mean():.3f} +/- {auc_scores.std():.3f}")
display(term_table.head(15), term_table.tail(15))

out = pd.concat([df, features.drop(columns=["high_rating"])], axis=1)
out.to_csv(PROC / "womens_clothing_product_text_features.csv", index=False)
correlations.to_csv(REPORTS / "task_14_product_text_analysis.csv", index=False)
term_table.to_csv(REPORTS / "task_14_product_text_terms.csv", index=False)
status = pd.DataFrame([{
    "task": "Task 14 - Product Text NLP Analysis", "status": "completed",
    "run_at": datetime.now().isoformat(timespec="seconds"), "input_file": DATA_PATH.name,
    "rows_in": len(df), "rows_with_text": int((text.str.len() > 0).sum()),
    "cv_roc_auc_mean": auc_scores.mean(), "cv_roc_auc_std": auc_scores.std(),
    "output_dataset": "womens_clothing_product_text_features.csv",
}])
status.to_csv(REPORTS / "task_14_analysis_status.csv", index=False)
display(status.T)

Text-only 5-fold ROC-AUC: 0.736 +/- 0.013


,term,rating_association
2935,print,1.269939
820,cotton,1.159575
509,camouflage,1.149824
1173,dress women,1.058306
1921,home,0.942439
385,bodycon,0.941933
826,cotton dress,0.940547
2021,jeans,0.851155
1754,funny,0.832732
2890,pocket,0.795285


,term,rating_association
2782,party dress,-0.797867
2094,lace,-0.810701
4178,summer plus,-0.819106
2836,plage,-0.877725
2568,neck women,-0.904182
4472,tops loose,-0.910695
1321,en,-0.929909
432,bra,-0.948206
2435,mini,-0.949318
235,beach,-0.984332


,0
task,Task 14 - Product Text NLP Analysis
status,completed
run_at,2026-11-20T15:11:54
input_file,womens_clothing_final.csv
rows_in,1539
rows_with_text,1539
cv_roc_auc_mean,0.73592
cv_roc_auc_std,0.012927
output_dataset,womens_clothing_product_text_features.csv
